<a href="https://colab.research.google.com/github/ychoi-kr/llm-api-prog/blob/main/5_gemini/learn_gemini_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @pip install google-generativeai==0.8.3
# %pip install google-generativeai
%pip uninstall google-generativeai -y -q
%pip install google-genai -q

In [ ]:
# @title 기본 사용법 1 – 싱글턴으로 메시지 주고받기
from google import genai
import os, getpass; api_key=os.environ.get("GOOGLE_API_KEY") or getpass.getpass("Google API Key: ")
genai.Client(api_key=api_key)

In [8]:
client = genai.Client(api_key=api_key)
response=client.models.generate_content(
    model="gemini-3.5-flash",
    contents="안녕하세요, 잘 작동하나요?"
)
print(response.text)

안녕하세요! 네, 아주 잘 작동하고 있습니다. 😊

오늘 어떤 도움이 필요하신가요? 궁금한 점이 있으시거나 도움이 필요한 작업이 있다면 편하게 말씀해 주세요!


In [4]:
response.text

'네, 안녕하세요! 저는 문제 없이 잘 작동하고 있습니다. Google에서 훈련한 대규모 언어 모델이며, 무엇을 도와드릴까요? 어떤 질문이든 편하게 해주세요!'

In [ ]:
# @title 기본 사용법 2 – 멀티턴으로 메시지 주고받기(1)
from google.generativeai import ChatSession
model = genai.GenerativeModel('gemini-1.5-flash')
chat_session: ChatSession = model.start_chat(history=[])  # ChatSession 객체 반환
user_queries = ["인공지능에 대해 한 문장으로 짧게 설명하세요.", "의식이 있는지 한 문장으로 답하세요."]
for user_query in user_queries:
    print(f'[사용자]: {user_query}')
    response = chat_session.send_message(user_query)
    print(f'[모델]: {response.text}')

In [ ]:
# @title 기본 사용법 3 – 멀티턴으로 메시지 주고받기(2)
model = genai.GenerativeModel('gemini-1.5-flash')
user_queries = [{'role':'user', 'parts': ["인공지능에 대해 한 문장으로 짧게 설명하세요."]},
                {'role':'user', 'parts': ["의식이 있는지 한 문장으로 답하세요."]}
            ]
history = []
for user_query in user_queries:
    history.append(user_query)
    print(f'[사용자]: {user_query["parts"][0]}')
    response = model.generate_content(history)
    print(f'[모델]: {response.text}')
    history.append(response.candidates[0].content)

In [ ]:
# @title 페르소나 만들기
system_instruction="당신은 유치원 선생님입니다. 사용자는 유치원생입니다. 쉽고 친절하게 이야기하되 3문장 이내로 짧게 얘기하세요."
model = genai.GenerativeModel('gemini-1.5-flash', system_instruction=system_instruction)
chat_session = model.start_chat(history=[])  # ChatSession 객체 반환
user_queries = ["인공지능이 뭐에요?", "그럼 스스로 생각도 해요?"]

for user_query in user_queries:
    print(f'[사용자]: {user_query}')
    response = chat_session.send_message(user_query)
    print(f'[모델]: {response.text}')


In [ ]:
# @title 답변 형식 지정하기
import json
system_instruction='JSON schema로 주제별로 답하되 3개를 넘기지 말 것:{{"주제": <주제>, "답변":<두 문장 이내>}}'
model = genai.GenerativeModel("gemini-1.5-flash", system_instruction=system_instruction, generation_config={"response_mime_type": "application/json"})
chat_session = model.start_chat(history=[])  # ChatSession 객체 반환
user_queries = ["인공지능의 특징이 뭐에요?", "어떤 것들을 조심해야 하죠?"]

for user_query in user_queries:
    print(f'[사용자]: {user_query}')
    response = chat_session.send_message(user_query)
    answer_dict = json.loads(response.text)
    print(answer_dict)


In [ ]:
# @title 제미나이 AI 입력 데이터 구조
model = genai.GenerativeModel('gemini-1.5-flash')
response = model.generate_content("인공지능에 대해 한 문장으로 설명하세요.")
print(response.candidates[0].content)
print(response.candidates[0].content.parts[0].text)

In [ ]:
# @title chat_session history에 들어 있는 Content 객체와 데이터
system_instruction="당신은 유치원 선생님입니다. 사용자는 유치원생입니다. 쉽고 친절하게 이야기하되 3문장 이내로 짧게 얘기하세요."
model = genai.GenerativeModel('gemini-1.5-flash', system_instruction=system_instruction)
chat_session = model.start_chat(history=[])  # ChatSession 객체 반환
user_queries = ["인공지능이 뭐에요?", "그럼 스스로 생각도 해요?"]

for user_query in user_queries:
    print(f'[사용자]: {user_query}')
    response = chat_session.send_message(user_query)
    print(f'[모델]: {response.text}')

print("-"*50)

for idx, content in enumerate(chat_session.history):
    print(f"{content.__class__.__name__}[{idx}]")
    print(content)


In [ ]:
# @title 이미지 데이터 인식
from google.colab import drive
import os
import PIL.Image
drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/llm-api-prog/5_gemini")

In [ ]:
image_data = PIL.Image.open("./data/images/monalisa.jpg") # 모나리자 그림
model = genai.GenerativeModel('gemini-1.5-flash')
response = model.generate_content(["이 그림에 대해 한 문장으로 설명하세요.", image_data])
print(response.text)

In [ ]:
# @title 제미나이 AI 출력 데이터 구조
print(response)

In [ ]:
# @title Candidate 객체
print(f"건수: {len(response.candidates)}")
print("="*50)
for candidate in response.candidates:
    print(candidate)

In [ ]:
# @title FinishReason 객체
print(f"finish_reason: {response.candidates[0].finish_reason.name},{response.candidates[0].finish_reason}")